## Setup: install GeoPandas (once)

This is the same as `%pip install pandas` in Lab 1. It can take a minute or two. If the next cell then says it cannot find geopandas, restart the kernel (the Restart button at the top of the notebook) and run it again. If the install itself fails, show your TA the last red line.

In [1]:
%pip install geopandas


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import geopandas as gpd
import json, os
print("GeoPandas", gpd.__version__, "is ready")

GeoPandas 1.1.4 is ready


*Importing the actual data*
 to get an idea of how to go about it moving foward 

In [26]:
gdf = gpd.read_file("../Data/Aires_repartition_MT.geojson")

Figuring out the size and main content of the file because she huge bro 

In [8]:
print("Total habitat features:", len(gdf))
print(gdf.geom_type.value_counts())
print(gdf["NOM_ANGLA"].dropna().unique()[:20])

Total habitat features: 69
MultiPolygon    62
Polygon          7
Name: count, dtype: int64
<StringArray>
[      'Long-tailed weasel',             'Least weasel',
                   'Muskox', 'Southern red-backed vole',
              'Meadow Vole',                'Rock Vole',
            'Woodland Vole',     'Northern bog Lemming',
     'Southern bog Lemming',                'Wolverine',
         'Woodland caribou',          'American beaver',
        'White-tailed deer',        'Silver-haired bat',
                'Hoary bat',  'Northern long-eared bat',
 'Eastern small-footed bat',                  'Red bat',
          'Star-nosed Mole',                   'coyote']
Length: 20, dtype: str


What organization and cordinate system is used in the data 

In [9]:
print("CRS:", gdf.crs)
print("Columns:", gdf.columns.tolist())

CRS: EPSG:32198
Columns: ['DESC_ENTIT', 'PRODUCTEUR', 'NOM_FRANCA', 'NOM_ANGLA', 'NOM_SCIENT', 'FAMILLE', 'GRAND_GROUPE', 'DATE_MAJ', 'Shape_Length', 'Shape_Area', 'geometry']


Simplifying the data first as there is a lot info that is not relevant to what I want to show 

In [10]:
gdf = gdf[
    [
        "NOM_FRANCA",
        "NOM_ANGLA",
        "NOM_SCIENT",
        "FAMILLE",
        "Shape_Length",
        "Shape_Area",
        "geometry"
    ]
].copy()

print(gdf.columns.tolist())

['NOM_FRANCA', 'NOM_ANGLA', 'NOM_SCIENT', 'FAMILLE', 'Shape_Length', 'Shape_Area', 'geometry']


Trial with a smaller data set as big one did not load so gonna attempt it with one species

In [11]:
print(gdf["NOM_ANGLA"].dropna().sort_values().unique())

<StringArray>
[         'American beaver',          'American marten',
       'American porcupine',    'American red squirrel',
     'American water shrew',             'Arctic Shrew',
               'Arctic fox',              'Arctic hare',
            'Big brown bat',              'Canada lynx',
               'Deer Mouse',         'Eastern chipmunk',
       'Eastern cottontail',    'Eastern grey squirrel',
     'Eastern heather vole',      'Eastern pipistrelle',
 'Eastern small-footed bat',                   'Ermine',
                   'Fisher',              'Gaspé Shrew',
                 'Grey fox',                'Grey wolf',
        'Hairy-tailed Mole',                'Hoary bat',
              'House Mouse',           'Least chipmunk',
             'Least weasel',         'Little brown bat',
       'Long-tailed weasel',             'Masked Shrew',
              'Meadow Vole',     'Meadow jumping Mouse',
                     'Mink',                    'Moose',
                 

In [12]:
species_to_keep = ["Moose", "Woodland caribou"]

gdf_test = gdf[gdf["NOM_ANGLA"].isin(species_to_keep)].copy()

print("Features in test dataset:", len(gdf_test))
print(gdf_test["NOM_ANGLA"].value_counts())

Features in test dataset: 2
NOM_ANGLA
Woodland caribou    1
Moose               1
Name: count, dtype: int64


### Recipe 4. Too big: filter first

Your MVP needs one borough, one year or one category, not the whole province. Filter on a column, or clip to a box. Count before and after, and put the numbers in your README.

I did this seperatly and made a seperate kind of variable gdf_test so that way I can add more data later i I want to 

In [27]:

print("before:", len(gdf))

species_to_keep = ["Moose", "Woodland caribou"]
gdf_test = gdf[gdf["NOM_ANGLA"].isin(species_to_keep)].copy()  #filter by a value

print("after:", len(gdf_test))

before: 69
after: 2


In [37]:
print(gdf_test.crs)

EPSG:4326


### Recipe 3. A shapefile, GeoPackage or GeoJSON (points, lines or areas)

A shapefile is several files with the same name (`.shp`, `.shx`, `.dbf`, `.prj` and more); keep them together in `data/` and point at the `.shp`. GeoPandas reads the coordinate system from the `.prj`. Lines and areas are often far more detailed than a web map needs: `simplify` removes points closer together than the tolerance, **in the file's own units**, so do it before converting to degrees.

In [38]:
gdf = gpd.read_file("../Data/Aires_repartition_MT.geojson")      # or .gpkg, or .geojson
print(gdf.crs, len(gdf), sorted(gdf.geom_type.unique()))
print(gdf.columns.tolist())
gdf_test = gdf_test[["NOM_ANGLA", "NOM_FRANCA", "NOM_SCIENT", "FAMILLE", "geometry"]]# # Lines or areas only: simplify to about 5 metres if the file is in metres (skip if gdf.crs says degrees)
gdf_test["geometry"] = gdf_test.geometry.simplify(3)
gdf_test = gdf_test.to_crs("EPSG:4326")
gdf_test.to_file("../Data/Aires_repartition_MT_simplified.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

EPSG:32198 69 ['MultiPolygon', 'Polygon']
['DESC_ENTIT', 'PRODUCTEUR', 'NOM_FRANCA', 'NOM_ANGLA', 'NOM_SCIENT', 'FAMILLE', 'GRAND_GROUPE', 'DATE_MAJ', 'Shape_Length', 'Shape_Area', 'geometry']


In [33]:
gdf_test.shape

(2, 5)

### The check cell you will reuse for your own data

Change only the four settings at the top when you use it on your project. It reads the file back the way the browser will.

In [39]:
# ---- settings: change these for your own data ----
FILE = "../Data/Aires_repartition_MT_simplified.geojson"
WEST, EAST, SOUTH, NORTH = -79.76, 44.99, -56.93, 62.59  # a box around your area, in degrees
EXPECTED = 2          # how many features you expect (rows after your filters)
POPUP_FIELDS = ["NOM_ANGLA", "NOM_FRANCA", "NOM_SCIENT", "FAMILLE"]  # which fields you want to show in the popup
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['MultiPolygon']
All checks passed
